# HelloML — Handwritten Digit OCR (DIDA)

Google Colab–oriented notebook. One **Pipeline + GridSearchCV** covers all models in a single large `grid` object.

| Mode | Value | What it does |
|------|-------|----------------|
| Train | `1` | Load DIDA → preprocess → save PNGs → **one** GridSearchCV → evaluate → save joblib |
| Load  | `2` | Load joblib artifacts → evaluate / plot (no training) |

**Drive layout (after `os.chdir`):**
```text
OCR_HelloML Project/
├── Dataset/
│   ├── DIDA/0..9
│   └── Processed_Data/0..9
├── Saved Variables & Processed Dataset/   ← *.joblib (incl. grid.joblib)
├── helloml_pipeline.py
└── OCR_HelloML_Complete.ipynb
```


## 0. Mount Drive & config


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Project root on Google Drive
ROOT_DIR = Path("/content/drive/MyDrive/OCR_HelloML Project")
os.chdir(ROOT_DIR)
print("cwd:", Path.cwd())

from helloml_pipeline import (
    load_dataset, binarize_center_resize, normalize_flatten_split,
    save_processed_images, get_pipeline_and_param_grid, summarize_cv_results,
    SCORING_METRICS, save_artifacts, load_artifacts, artifacts_exist, ARTIFACT_KEYS,
)

# ============================================================
# CONFIG
# ============================================================
RUN_MODE = 1                # 1 = train  |  2 = load
FILES_PER_FOLDER = 1000     # e.g. 200 for a quick dry-run

DATASET_DIR = Path.cwd() / "Dataset"
DIDA_DIR = DATASET_DIR / "DIDA"
PROCESSED_DIR = DATASET_DIR / "Processed_Data"
ARTIFACTS_DIR = Path.cwd() / "Saved Variables & Processed Dataset"
# ============================================================

if RUN_MODE not in (1, 2):
    raise ValueError("RUN_MODE must be 1 (train) or 2 (load)")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

mode_name = {1: "train", 2: "load"}[RUN_MODE]
print(f"RUN_MODE      = {RUN_MODE}  ({mode_name})")
print(f"DIDA_DIR      = {DIDA_DIR}")
print(f"PROCESSED_DIR = {PROCESSED_DIR}")
print(f"ARTIFACTS_DIR = {ARTIFACTS_DIR}")


## 1. Data (train or load)


In [ ]:
if RUN_MODE == 2:
    if not artifacts_exist(ARTIFACTS_DIR):
        raise FileNotFoundError(
            f"No artifacts in {ARTIFACTS_DIR}. Run once with RUN_MODE = 1."
        )
    print("Loading data artifacts...")
    load_artifacts(ARTIFACTS_DIR, globals(), keys=[
        "X_train", "X_test", "y_train", "y_test", "X_proc", "y"
    ])
    X_raw = None
elif RUN_MODE == 1:
    if not DIDA_DIR.exists():
        raise FileNotFoundError(f"DIDA not found at {DIDA_DIR}")
    X_raw, y = load_dataset(DIDA_DIR, files_per_folder=FILES_PER_FOLDER)
    X_proc = binarize_center_resize(X_raw)
    save_processed_images(X_proc, y, PROCESSED_DIR, clear=True)
    X_train, X_test, y_train, y_test = normalize_flatten_split(X_proc, y)


## 2. Preview samples


In [ ]:
if X_proc is not None:
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    for digit, ax in enumerate(axes.ravel()):
        idxs = np.where(y == digit)[0]
        ax.imshow(X_proc[idxs[0]], cmap="gray")
        ax.set_title(str(digit))
        ax.axis("off")
    plt.suptitle("Preprocessed samples", fontweight="bold")
    plt.tight_layout()
    plt.show()


## 3. Single Pipeline GridSearchCV (or load)

One `Pipeline` + one `param_grid` list → **one** `grid` object for all model families
(NaiveBayes, LinearReg OvA, LogisticReg, MLP).


In [ ]:
if RUN_MODE == 2:
    print("Loading model artifacts (skip training)...")
    load_artifacts(ARTIFACTS_DIR, globals(), keys=[
        "grid", "best_estimator", "df_cv_all", "df_cv_best", "df_test"
    ])
    display(df_cv_best)
    print("Best params (overall):", grid.best_params_)
    print("Best F1 (CV):", grid.best_score_)
else:
    pipe, param_grid = get_pipeline_and_param_grid()
    print("=" * 60)
    print("PHASE 1: single GridSearchCV over Pipeline")
    print("=" * 60)
    print(f"Param grid blocks: {len(param_grid)}")

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring=SCORING_METRICS,
        refit="f1",
        n_jobs=-1,
        return_train_score=False,
        verbose=1,
    )
    t0 = time.time()
    grid.fit(X_train, y_train)
    print(f"Done in {time.time() - t0:.1f}s")
    print("Best params:", grid.best_params_)
    print("Best F1 (CV):", grid.best_score_)

    best_estimator = grid.best_estimator_  # Pipeline with winning clf
    df_cv_all, df_cv_best = summarize_cv_results(grid)
    display(df_cv_best)

    print("\nSaving artifacts (including full grid)...")
    save_artifacts(ARTIFACTS_DIR, globals())


## 4. CV confusion matrix (overall best model)


In [ ]:
# Use the single best Pipeline from GridSearchCV
y_pred_cv = cross_val_predict(grid.best_estimator_, X_train, y_train, cv=5)
cm = confusion_matrix(y_train, y_pred_cv)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
ax.set_title("CV Confusion Matrix — best Pipeline")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()


## 5. Test evaluation


In [ ]:
print("=" * 60)
print("PHASE 2: Test set (best Pipeline from grid)")
print("=" * 60)

y_pred = grid.best_estimator_.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f}")
target_names = [f"Digit {i}" for i in range(10)]
print(classification_report(y_test, y_pred, target_names=target_names, digits=3))

df_test = pd.DataFrame([{"Model": "Best Pipeline (grid)", "Test Accuracy": acc}])
display(df_test)

if RUN_MODE == 1:
    joblib.dump(df_test, ARTIFACTS_DIR / "df_test.joblib")
    print(f"Saved df_test -> {ARTIFACTS_DIR / 'df_test.joblib'}")


## 6. Charts (CV best per family)


In [ ]:
colors = ["#7e57c2", "#26a69a", "#42a5f5", "#ef5350"]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(df_cv_best["Model"], df_cv_best["Mean F1"], color=colors[:len(df_cv_best)])
ax.set_title("Best CV F1 per model family", fontweight="bold")
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.6)
for b in bars:
    h = b.get_height()
    ax.annotate(f"{h:.1%}", (b.get_x() + b.get_width() / 2, h),
                ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Artifacts listing


In [ ]:
print("Artifacts folder:", ARTIFACTS_DIR)
for p in sorted(ARTIFACTS_DIR.glob("*.joblib")):
    print(f"  {p.name:30s}  {p.stat().st_size / 1024 / 1024:6.2f} MB")

print("\nProcessed images:", PROCESSED_DIR)
if PROCESSED_DIR.exists():
    for d in range(10):
        folder = PROCESSED_DIR / str(d)
        n = len(list(folder.glob("*"))) if folder.exists() else 0
        print(f"  digit {d}: {n} files")


## Workflow

```text
Colab:  mount Drive → chdir to "OCR_HelloML Project"

1st time:  RUN_MODE = 1
           → Dataset/Processed_Data/0..9/*.png
           → Saved Variables & Processed Dataset/grid.joblib   (full GridSearchCV)
           → best_estimator.joblib, df_*.joblib, arrays...

Later:     RUN_MODE = 2
           → load grid / data, plots & reports only
           → grid.best_params_, grid.cv_results_, grid.best_estimator_ available
```
